# เทรนโมเดล GRU พยากรณ์ราคาน้ำมันดิบ WTI (CL=F)
ข้อมูล: `crude_oil_15y.csv` (ราคาปิดรายวันย้อนหลัง 15 ปีจาก Yahoo Finance)

ขั้นตอน: ติดตั้ง → โหลดข้อมูล → ทำความสะอาด → สร้างข้อมูลแบบลำดับ (sequence) → แบ่ง Train/Val/Test → เทรน GRU → วัดความแม่นยำ (MAE, RMSE, MAPE, Accuracy) เทียบกับ Baseline → ทดสอบพยากรณ์ล่วงหน้าหลายวัน → เทรนใหม่ด้วยข้อมูลทั้งหมด → พยากรณ์อนาคต → เซฟโมเดล

**กดรันทีละ cell จากบนลงล่าง** (Shift + Enter) ถ้าใช้ Colab แนะนำเปิด GPU: Runtime → Change runtime type → GPU (ไม่เปิดก็รันได้ แค่ช้ากว่า)

## 1. ติดตั้งไลบรารี (รันครั้งแรกครั้งเดียว — Colab มี TensorFlow อยู่แล้ว)

In [ ]:
%pip install -q tensorflow pandas numpy matplotlib scikit-learn

## 2. Import และตั้งค่า

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import mean_absolute_error, mean_squared_error

CSV_PATH    = "crude_oil_15y.csv"
MODEL_DIR   = "MODEL"
MODEL_PATH  = os.path.join(MODEL_DIR, "oil_gru_model.keras")
PARAMS_PATH = os.path.join(MODEL_DIR, "oil_gru_params.json")
FORECAST_PATH = os.path.join(MODEL_DIR, "oil_gru_forecast.csv")

LOOKBACK  = 60      # ใช้ข้อมูลย้อนหลัง 60 วันทำการ เพื่อพยากรณ์วันถัดไป
TEST_DAYS = 365     # ใช้ข้อมูล 1 ปีล่าสุดเป็นชุดทดสอบ
VAL_RATIO = 0.15    # แบ่ง 15% ท้ายของ Train ไว้เป็น Validation (ใช้หยุดเทรนเมื่อไม่ดีขึ้น)
CLIP_SD   = 5       # ตัดผลตอบแทนรายวันที่เกิน ±5 เท่าของส่วนเบี่ยงเบน (กันวันผิดปกติ)
MIN_EPOCHS = 15     # เทรนอย่างน้อย 15 epoch ก่อนให้ EarlyStopping ทำงาน
HORIZON   = 7       # พยากรณ์ล่วงหน้า 7 วันทำการ (ตามขอบเขตข้อ 2.3.6.2)
EPOCHS    = 100
BATCH     = 32
SEED      = 42

os.makedirs(MODEL_DIR, exist_ok=True)
keras.utils.set_random_seed(SEED)     # ให้ผลลัพธ์ซ้ำได้ใกล้เคียงเดิม
plt.rcParams["figure.figsize"] = (14, 5)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 3. โหลดข้อมูล
> ถ้าใช้ **Google Colab** ให้รัน cell ข้างล่างเพื่ออัปโหลดไฟล์ CSV (รันบนเครื่องตัวเองข้ามได้)

In [ ]:
try:
    from google.colab import files
    files.upload()
except ImportError:
    print("ไม่ได้รันบน Colab — ข้าม cell นี้ได้")

In [ ]:
raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
print("จำนวนแถว:", len(raw))
raw.head()

## 4. ทำความสะอาดข้อมูล

In [ ]:
df = raw.copy()

# 4.1 แปลงชนิดข้อมูล และตัดคอลัมน์ที่ไม่ใช้
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
df = df[["Date", "Close"]]

# 4.2 ตัดแถวว่าง/วันที่ซ้ำ แล้วเรียงตามวันที่
before = len(df)
df = df.dropna().drop_duplicates(subset="Date", keep="last").sort_values("Date")
print("ตัดแถวว่าง/ซ้ำ:", before - len(df))

# 4.3 ตัดแถวของวันนี้ (ตลาดยังไม่ปิด)
today = pd.Timestamp.today().normalize()
print("ตัดแถววันนี้:", (df["Date"] >= today).sum())
df = df[df["Date"] < today]

# 4.4 ปัดทศนิยม 2 ตำแหน่ง
df["Close"] = df["Close"].round(2)

# 4.5 ราคาติดลบ/เป็นศูนย์ (20 เม.ย. 2020 WTI ติดลบ −37.63) คำนวณ log ไม่ได้ จึงตัดออก
bad = df["Close"] <= 0
print("ราคา <= 0 ที่ตัดออก:", bad.sum(), df.loc[bad, "Date"].dt.date.tolist())
df = df[~bad]

df = df.reset_index(drop=True)
print("เหลือ", len(df), "แถว ตั้งแต่", df["Date"].min().date(), "ถึง", df["Date"].max().date())

In [ ]:
plt.plot(df["Date"], df["Close"])
plt.title("WTI Crude Oil (CL=F) Close Price"); plt.grid(alpha=.3); plt.show()

## 5. แปลงเป็นผลตอบแทนรายวัน (log return)
GRU จะเรียนจาก **% การเปลี่ยนแปลงรายวัน** แทนราคาตรง ๆ เพราะ:
- ราคาทอง/น้ำมันช่วงทดสอบสูงกว่าช่วงที่เคยเห็นตอนเทรน ถ้าใช้ราคาตรง ๆ โมเดลจะพยากรณ์เกินช่วงที่รู้จักไม่ได้
- ผลตอบแทนรายวันมีสเกลคงที่ตลอด 15 ปี โมเดลเรียนง่ายกว่า

จากนั้นแปลงกลับเป็นราคา: `ราคาพรุ่งนี้ = ราคาวันนี้ × exp(ผลตอบแทนที่พยากรณ์)`

In [ ]:
df["ret"] = np.log(df["Close"]).diff()
df = df.dropna().reset_index(drop=True)

prices = df["Close"].values
rets   = df["ret"].values
dates  = df["Date"].values
df.tail()

## 6. แบ่ง Train / Validation / Test และ Standardize
- แบ่งตามเวลา (ห้ามสุ่ม): Test = 1 ปีล่าสุด, Validation = 15% ท้ายของส่วนที่เหลือ
- ตัดผลตอบแทนวันที่ผิดปกติมาก ๆ (เกิน ±5 SD) ไม่ให้ครอบงำการเทรน
- Standardize ด้วยค่าเฉลี่ย/ส่วนเบี่ยงเบนของ **Train เท่านั้น** (กันข้อมูลอนาคตรั่วเข้าโมเดล)

In [ ]:
test_start = np.searchsorted(dates, dates[-1] - np.timedelta64(TEST_DAYS, "D"), side="right")
val_start  = int(test_start * (1 - VAL_RATIO))

# ตัดค่าผลตอบแทนที่สุดโต่ง (เช่นช่วงโควิด 2020) ให้อยู่ในช่วง ±CLIP_SD เท่าของส่วนเบี่ยงเบน
# ไม่งั้นไม่กี่วันที่ผิดปกติจะครอบงำการเทรน
sd0 = rets[:val_start].std()
n_clip = (np.abs(rets) > CLIP_SD * sd0).sum()
rets_c = np.clip(rets, -CLIP_SD * sd0, CLIP_SD * sd0)
print(f"ตัดค่าผลตอบแทนสุดโต่ง (> {CLIP_SD} SD): {n_clip} วัน")

MU, SD = rets_c[:val_start].mean(), rets_c[:val_start].std()
z = (rets_c - MU) / SD

def make_xy(idx_from, idx_to):
    # X = ผลตอบแทน LOOKBACK วันก่อนหน้า, y = ผลตอบแทนวันที่ i
    X, y, idx = [], [], []
    for i in range(max(idx_from, LOOKBACK), idx_to):
        X.append(z[i - LOOKBACK:i]); y.append(z[i]); idx.append(i)
    return np.array(X)[..., None], np.array(y), np.array(idx)

X_train, y_train, i_train = make_xy(0, val_start)
X_val,   y_val,   i_val   = make_xy(val_start, test_start)
X_test,  y_test,  i_test  = make_xy(test_start, len(z))

print(f"Train: {X_train.shape}  ({str(dates[i_train[0]])[:10]} ถึง {str(dates[i_train[-1]])[:10]})")
print(f"Val  : {X_val.shape}  ({str(dates[i_val[0]])[:10]} ถึง {str(dates[i_val[-1]])[:10]})")
print(f"Test : {X_test.shape}  ({str(dates[i_test[0]])[:10]} ถึง {str(dates[i_test[-1]])[:10]})")

## 7. สร้างและเทรนโมเดล GRU

In [ ]:
def build_model():
    m = keras.Sequential([
        keras.Input(shape=(LOOKBACK, 1)),
        keras.layers.GRU(64, return_sequences=True),
        keras.layers.Dropout(0.2),
        keras.layers.GRU(32),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1),
    ])
    m.compile(optimizer=keras.optimizers.Adam(5e-4), loss="mse")   # huber ทนต่อวันที่ราคากระชากแรง
    return m

model = build_model()
model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True,
                                  start_from_epoch=MIN_EPOCHS),   # ไม่หยุดก่อน MIN_EPOCHS
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5),
]
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=EPOCHS, batch_size=BATCH, callbacks=callbacks, verbose=2)
val_loss = np.array(history.history["val_loss"])
BEST_EPOCH = MIN_EPOCHS + int(np.argmin(val_loss[MIN_EPOCHS:])) + 1 if len(val_loss) > MIN_EPOCHS else len(val_loss)
print("เทรนไป", len(val_loss), "epoch | epoch ที่ดีที่สุด (หลัง MIN_EPOCHS):", BEST_EPOCH)

In [ ]:
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.axvline(BEST_EPOCH - 1, color="gray", ls="--", label="best epoch")
plt.title("Training history"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 8. พยากรณ์ชุด Test (ล่วงหน้า 1 วัน)
ทุกวันในชุดทดสอบ ใช้ราคาจริง 60 วันก่อนหน้า พยากรณ์ราคาวันถัดไป

In [ ]:
pred_ret = model.predict(X_test, verbose=0).ravel() * SD + MU
prev_price = prices[i_test - 1]
result = pd.DataFrame({
    "ds": dates[i_test],
    "y": prices[i_test],
    "yhat": prev_price * np.exp(pred_ret),
    "naive": prev_price,                       # Baseline: ราคาพรุ่งนี้ = ราคาวันนี้
})
result.head()

## 9. วัดความแม่นยำ: MAE, RMSE, MAPE, Accuracy
- **MAE** ผิดเฉลี่ยกี่ดอลลาร์ | **RMSE** ลงโทษวันที่ผิดมากหนักกว่า | **MAPE** ผิดเฉลี่ยกี่ %
- **Accuracy** = 100 − MAPE
- **Naive** = เดาว่าราคาพรุ่งนี้เท่ากับวันนี้ ถ้า GRU แพ้ Naive แปลว่าโมเดลยังไม่ได้ช่วยอะไร
- **Direction Accuracy** = ทายถูกกี่ % ว่าพรุ่งนี้ราคาขึ้นหรือลง (50% = เท่ากับโยนเหรียญ)

In [ ]:
def evaluate(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAPE (%)": mape,
            "Accuracy (%)": 100 - mape}

metrics = pd.DataFrame({"GRU": evaluate(result["y"], result["yhat"]),
                        "Naive (ราคาวันก่อน)": evaluate(result["y"], result["naive"])}).round(2)
print("พยากรณ์ล่วงหน้า 1 วัน (ชุดทดสอบ 1 ปี):")
print(metrics)

actual_up = result["y"].values > result["naive"].values
pred_up   = result["yhat"].values > result["naive"].values
dir_acc = (actual_up == pred_up).mean() * 100
print(f"\nDirection Accuracy (ทายขึ้น/ลงถูก): {dir_acc:.2f}%")
print(f">>> ความแม่นยำของโมเดล (Accuracy) = {metrics.loc['Accuracy (%)', 'GRU']:.2f}%")

In [ ]:
plt.plot(result["ds"], result["y"], label="Actual", color="black")
plt.plot(result["ds"], result["yhat"], label="GRU (1-day ahead)", color="tab:orange", alpha=.8)
m = metrics["GRU"]
plt.title(f"Test 1-day ahead: MAE={m['MAE']:.2f}  RMSE={m['RMSE']:.2f}  MAPE={m['MAPE (%)']:.2f}%  Accuracy={m['Accuracy (%)']:.2f}%")
plt.legend(); plt.grid(alpha=.3); plt.show()

## 9.1 ทดสอบพยากรณ์ล่วงหน้า 7 วัน (เทียบกับไฟล์ Prophet ได้)
พยากรณ์ล่วงหน้า 1 วันจะได้ค่าดีเสมอ เพราะรู้ราคาจริงเมื่อวาน การใช้งานจริงมักต้องพยากรณ์หลายวัน
ตรงนี้จะเริ่มจากหลายจุดในชุดทดสอบ (ทุก ๆ 7 วัน) แล้วให้ GRU พยากรณ์ต่อกันเอง 7 วัน โดยไม่รู้ราคาจริง (recursive) แล้วเฉลี่ยค่า MAE, RMSE, MAPE

In [ ]:
def forecast_recursive(m, z_hist, last_price, steps):
    window = list(z_hist[-LOOKBACK:])
    out, price = [], last_price
    for _ in range(steps):
        r = m.predict(np.array(window[-LOOKBACK:])[None, :, None], verbose=0)[0, 0]
        window.append(r)
        price = price * np.exp(r * SD + MU)
        out.append(price)
    return np.array(out)

rows = []
for start in range(test_start, len(z) - HORIZON + 1, HORIZON):
    y_true = prices[start:start + HORIZON]
    p_gru  = forecast_recursive(model, z[:start], prices[start - 1], HORIZON)
    p_nv   = np.full(HORIZON, prices[start - 1])
    g, n = evaluate(y_true, p_gru), evaluate(y_true, p_nv)
    rows.append({"start": str(dates[start])[:10], **{f"GRU {k}": v for k, v in g.items()}, **{f"Naive {k}": v for k, v in n.items()}})
    print(f"{str(dates[start])[:10]}  GRU MAPE {g['MAPE (%)']:6.2f}%   Naive MAPE {n['MAPE (%)']:6.2f}%")

ms = pd.DataFrame(rows)
keys = ["MAE", "RMSE", "MAPE (%)", "Accuracy (%)"]
metrics_h = pd.DataFrame({"GRU": [ms[f"GRU {k}"].mean() for k in keys],
                          "Naive (ราคาล่าสุด)": [ms[f"Naive {k}"].mean() for k in keys]}, index=keys).round(2)
print(f"\nค่าเฉลี่ยจาก {len(ms)} รอบ (พยากรณ์ล่วงหน้า {HORIZON} วัน):")
print(metrics_h)

## 10. เทรนใหม่ด้วยข้อมูลทั้งหมด แล้วพยากรณ์อนาคต
ใช้จำนวน epoch เท่ากับที่ดีที่สุดจากข้อ 7 เพื่อให้โมเดลรู้ราคาล่าสุด

In [ ]:
keras.utils.set_random_seed(SEED)
X_all, y_all, _ = make_xy(0, len(z))
final_model = build_model()
final_model.fit(X_all, y_all, epochs=BEST_EPOCH, batch_size=BATCH, verbose=0)

future_prices = forecast_recursive(final_model, z, prices[-1], HORIZON)
future_dates  = pd.bdate_range(df["Date"].max() + pd.Timedelta(days=1), periods=HORIZON)
forecast = pd.DataFrame({"ds": future_dates, "yhat": future_prices.round(2)})
forecast.head(10)

In [ ]:
hist = df[df["Date"] >= df["Date"].max() - pd.Timedelta(days=365)]
plt.plot(hist["Date"], hist["Close"], color="black", label="Actual")
plt.plot(forecast["ds"], forecast["yhat"], color="tab:orange", label="GRU forecast")
plt.title(f"GRU forecast next {HORIZON} business days"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 11. เซฟโมเดล ค่าที่ต้องใช้ และผลพยากรณ์

In [ ]:
final_model.save(MODEL_PATH)
with open(PARAMS_PATH, "w") as f:
    json.dump({"LOOKBACK": LOOKBACK, "MU": float(MU), "SD": float(SD), "CLIP": float(CLIP_SD * sd0),
               "last_date": str(df["Date"].max().date()), "last_price": float(prices[-1])}, f, indent=2)

forecast.to_csv(FORECAST_PATH, index=False)
metrics.to_csv(os.path.join(MODEL_DIR, "oil_gru_metrics_1day.csv"))
metrics_h.to_csv(os.path.join(MODEL_DIR, "oil_gru_metrics_{}day.csv".format(HORIZON)))

print("เซฟโมเดล       ->", MODEL_PATH)
print("เซฟค่า MU/SD   ->", PARAMS_PATH)
print("เซฟพยากรณ์     ->", FORECAST_PATH)

## 12. ทดสอบโหลดโมเดลกลับมาใช้

In [ ]:
loaded = keras.models.load_model(MODEL_PATH)
with open(PARAMS_PATH) as f:
    params = json.load(f)

check = forecast_recursive(loaded, z, params["last_price"], 5)
print("พยากรณ์ 5 วันข้างหน้าจากโมเดลที่โหลด:", check.round(2))